# Unified Evaluation: 4-Step Optimized vs Single Prompt

**8 metrics** compared across 3 models:
- Step 1.1: `F1_Participants`, `F1_Restaurants`, `Final_Restaurant_Match`
- Step 1.2: `F1_Suggestion`, `F1_Response`
- Step 2: `F1_Mentioned`
- Step 3: `F1_PerceptionTable` (micro F1, paper Eq.4)
- Step 4: `F1_InterpretationTable` (Positive-F1, paper Eq.5-6)

**4-step optimized** = best technique × best iteration (from `final_results.json`) 
**Single prompt** = 5 iterations averaged 
**Scope filtering**: Step 1 best output → participant/restaurant scope → applied to Steps 2-4

In [1]:
import json
import os
import glob
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.display import display, HTML

## Configuration

In [2]:
DATA_DIR = 'data'
GOLD_DIR = os.path.join(DATA_DIR, 'gold')

FOURSTEP_DIRS = {
    'GPT-5':     'results/gpt5/raw',
    'GPT-4o': 'results/gpt4o/raw',
}

SINGLE_DIRS = {
    'GPT-5':     'appendix/results/single_prompt/gpt5/raw',
    'GPT-4o': 'appendix/results/single_prompt/gpt4o/raw',
}

ITERATIONS = range(1, 6)
METRIC_NAMES = ['F1_Participants', 'F1_Restaurants', 'Final_Restaurant_Match',
                'F1_Suggestion', 'F1_Response', 'F1_Mentioned',
                'F1_PerceptionTable', 'F1_InterpretationTable']

print('Configuration loaded')

Configuration loaded


## Evaluation Functions

All functions match paper Eq.1-6 and existing pipeline logic.

In [3]:
def normalize_name(x):
    return str(x).strip().lower()

def f1_from_counts(tp, fp, fn):
    if tp == 0:
        return 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def set_f1(gold_list, pred_list):
    """Set-based F1 (Step 1.1 participants/restaurants)"""
    if not gold_list and not pred_list:
        return 1.0
    if not gold_list or not pred_list:
        return 0.0
    gold_set = {str(x).strip() for x in gold_list}
    pred_set = {str(x).strip() for x in pred_list}
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    return f1_from_counts(tp, fp, fn)

def table_f1(gold_table, pred_table, key_columns):
    """Row-level F1 for tables (Step 1.2 suggestion/response)"""
    if not gold_table and not pred_table:
        return 1.0
    if not gold_table or not pred_table:
        return 0.0
    def make_key(row):
        return ' '.join(str(row.get(c, '')).strip() for c in key_columns)
    gold_keys = set(make_key(r) for r in gold_table)
    pred_keys = set(make_key(r) for r in pred_table)
    tp = len(gold_keys & pred_keys)
    fp = len(pred_keys - gold_keys)
    fn = len(gold_keys - pred_keys)
    return f1_from_counts(tp, fp, fn)

def mentioned_f1(gold_table, pred_table, scope_restaurants):
    """
    Step 2: F1 on (participant, restaurant) pairs where mention='Mentioned',
    filtered by scope restaurants.
    """
    scope_r = {normalize_name(r) for r in scope_restaurants}

    def extract_mentioned_pairs(table):
        pairs = set()
        for row in table:
            if row.get('mention', '') == 'Mentioned':
                r = normalize_name(row.get('restaurant', ''))
                if r in scope_r:
                    pairs.add((row['participant'], r))
        return pairs

    gold_pairs = extract_mentioned_pairs(gold_table)
    pred_pairs = extract_mentioned_pairs(pred_table)

    if not gold_pairs and not pred_pairs:
        return 1.0
    tp = len(gold_pairs & pred_pairs)
    fp = len(pred_pairs - gold_pairs)
    fn = len(gold_pairs - pred_pairs)
    return f1_from_counts(tp, fp, fn)

def perception_micro_f1(gold_table, pred_table, scope_p, scope_r):
    """
    Step 3: Triplet micro F1 (paper Eq.4) with scope filtering.
    Compares (participant, restaurant) → sentiment/perception value.
    """
    scope_r_lower = {normalize_name(r) for r in scope_r}

    gold_map = {}
    for g in gold_table:
        if g['participant'] in scope_p and normalize_name(g['restaurant']) in scope_r_lower:
            key = (g['participant'], normalize_name(g['restaurant']))
            gold_map[key] = g.get('perception', g.get('sentiment', '')).strip()

    pred_map = {}
    for p in pred_table:
        if p['participant'] in scope_p and normalize_name(p['restaurant']) in scope_r_lower:
            key = (p['participant'], normalize_name(p['restaurant']))
            pred_map[key] = p.get('sentiment', p.get('perception', '')).strip()

    tp = fp = fn = 0
    for key, gv in gold_map.items():
        if key in pred_map:
            if gv == pred_map[key]:
                tp += 1
            else:
                fn += 1
        else:
            fn += 1
    for key in pred_map:
        if key not in gold_map:
            fp += 1
    return f1_from_counts(tp, fp, fn)

def parse_factors(s):
    """Factor string → set. 'A1,A2' → {'A1','A2'}, 'None' → set()"""
    if not s or str(s).strip().lower() == 'none':
        return set()
    return {f.strip() for f in str(s).split(',') if f.strip() and f.strip().lower() != 'none'}

def positive_f1_cell(gold_set, pred_set):
    """Paper Eq.6: per-cell factor set F1"""
    inter = len(gold_set & pred_set)
    prec = inter / len(pred_set) if pred_set else 0.0
    rec  = inter / len(gold_set) if gold_set else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

def integrate_tables(pref_tbl, cons_tbl):
    """
    Interpretation Table: preferences ∪ constraints per (participant, restaurant).
    Paper Step 4 unified evaluation.
    """
    union_map = {}
    for row in pref_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get('preferences', 'None')))
    for row in cons_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get('constraints', 'None')))
    return [
        {'participant': p, 'restaurant': r,
         'integrated': ','.join(sorted(factors)) if factors else 'None'}
        for (p, r), factors in union_map.items()
    ]

def interpretation_positive_f1(gold_pref, gold_cons, pred_pref, pred_cons, scope_p, scope_r):
    """
    Step 4: Positive-F1 on integrated Interpretation Table (paper Eq.5-6).
    Scope-filtered, then preferences ∪ constraints → integrated.
    """
    scope_r_lower = {normalize_name(r) for r in scope_r}

    def filt(table):
        return [t for t in table
                if t['participant'] in scope_p
                and normalize_name(t['restaurant']) in scope_r_lower]

    gp, gc = filt(gold_pref), filt(gold_cons)
    pp, pc = filt(pred_pref), filt(pred_cons)

    gold_int = integrate_tables(gp, gc)
    pred_int = integrate_tables(pp, pc)

    gold_map = {(it['participant'], normalize_name(it['restaurant'])): parse_factors(it['integrated']) for it in gold_int}
    pred_map = {(it['participant'], normalize_name(it['restaurant'])): parse_factors(it['integrated']) for it in pred_int}

    pos_sum = pos_cnt = 0
    for key, gold_set in gold_map.items():
        pred_set = pred_map.get(key, set())
        if gold_set:  # positive cell (D_positive)
            pos_sum += positive_f1_cell(gold_set, pred_set)
            pos_cnt += 1

    return pos_sum / pos_cnt if pos_cnt else 0.0

print('Evaluation functions defined')

Evaluation functions defined


## Data Loading Functions

In [4]:
def load_gold(log_name):
    gold_path = os.path.join(GOLD_DIR, log_name)
    gold = {}
    for key, fname in [('step1_1', 'step1_1_gold.json'), ('step1_2', 'step1_2_gold.json'),
                       ('step2', 'step2_gold.json'), ('step3', 'step3_gold.json'),
                       ('step4', 'step4_gold.json')]:
        fpath = os.path.join(gold_path, fname)
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                gold[key] = json.load(f)
    return gold

def load_4step_final(fourstep_dir, log_name):
    """Load the optimized (best technique × best iteration) result from final_results.json"""
    fpath = os.path.join(fourstep_dir, log_name, f'{log_name}_final_results.json')
    if not os.path.exists(fpath):
        return None
    with open(fpath, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_single_iterations(single_dir, log_name):
    """Load single prompt normalized results (up to 5 iterations)"""
    results = []
    log_dir = os.path.join(single_dir, log_name)
    if not os.path.isdir(log_dir):
        return results
    for it in ITERATIONS:
        fpath = os.path.join(log_dir, f'{log_name}_normalized_iter{it}.json')
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            results.append(data.get('result', data))
    return results

def get_conversation_list(fourstep_dir):
    if not os.path.isdir(fourstep_dir):
        return []
    return sorted([d for d in os.listdir(fourstep_dir)
                   if d.endswith('_log') and os.path.isdir(os.path.join(fourstep_dir, d))])

print('Data loading functions defined')

Data loading functions defined


## Evaluate All 8 Metrics

In [5]:
def evaluate_all_metrics(pred, gold, scope_p, scope_r):
    """
    Compute all 8 metrics for a single prediction result.
    pred: dict with participants, restaurant_brands, final_restaurant,
          suggestion_table, response_table, mentioned_table, sentiment_table,
          preference_table, constraint_table
    """
    metrics = {}

    # Step 1.1
    gold_s11 = gold.get('step1_1', {})
    metrics['F1_Participants'] = set_f1(
        gold_s11.get('participants', []),
        pred.get('participants', [])
    )
    metrics['F1_Restaurants'] = set_f1(
        gold_s11.get('restaurant_brands', []),
        pred.get('restaurant_brands', [])
    )
    metrics['Final_Restaurant_Match'] = (
        1.0 if gold_s11.get('final_restaurant', '').strip() == pred.get('final_restaurant', '').strip()
        else 0.0
    )

    # Step 1.2
    gold_s12 = gold.get('step1_2', {})
    metrics['F1_Suggestion'] = table_f1(
        gold_s12.get('suggestion_table', []),
        pred.get('suggestion_table', []),
        ['participant', 'suggestion_type']
    )
    metrics['F1_Response'] = table_f1(
        gold_s12.get('response_table', []),
        pred.get('response_table', []),
        ['participant', 'response_type']
    )

    # Step 2
    gold_s2 = gold.get('step2', {})
    metrics['F1_Mentioned'] = mentioned_f1(
        gold_s2.get('mentioned_table', []),
        pred.get('mentioned_table', []),
        scope_r
    )

    # Step 3
    gold_s3 = gold.get('step3', {})
    gold_s3_table = gold_s3.get('perception_table', gold_s3.get('sentiment_table', []))
    pred_s3_table = pred.get('sentiment_table', pred.get('perception_table', []))
    metrics['F1_PerceptionTable'] = perception_micro_f1(
        gold_s3_table, pred_s3_table, scope_p, scope_r
    )

    # Step 4
    gold_s4 = gold.get('step4', {})
    metrics['F1_InterpretationTable'] = interpretation_positive_f1(
        gold_s4.get('preference_table', []),
        gold_s4.get('constraint_table', []),
        pred.get('preference_table', []),
        pred.get('constraint_table', []),
        scope_p, scope_r
    )

    return metrics

print('evaluate_all_metrics() defined')

evaluate_all_metrics() defined


## Run Evaluation

In [6]:
all_rows = []

for model_label in FOURSTEP_DIRS:
    fourstep_dir = FOURSTEP_DIRS[model_label]
    single_dir = SINGLE_DIRS[model_label]
    logs = sorted([d for d in os.listdir(fourstep_dir)
                   if d.endswith('_log') and os.path.isdir(os.path.join(fourstep_dir, d))])
    print(f'\n{model_label}: {len(logs)} conversations')

    for log_name in logs:
        gold = load_gold(log_name)
        if not gold:
            continue
        final = load_4step_final(fourstep_dir, log_name)
        if not final:
            continue

        scope_p = set(final['step1']['participants'])
        scope_r = set(final['step1']['restaurant_brands'])

        # 4-step optimized
        pred_4step = {
            'participants': final['step1']['participants'],
            'restaurant_brands': final['step1']['restaurant_brands'],
            'final_restaurant': final['step1']['final_restaurant'],
            'suggestion_table': final['step1']['suggestion_table'],
            'response_table': final['step1']['response_table'],
            'mentioned_table': final['step2']['mentioned_table'],
            'sentiment_table': final['step3']['sentiment_table'],
            'preference_table': final['step4']['f1']['preference_table'],
            'constraint_table': final['step4']['f1']['constraint_table'],
        }
        m = evaluate_all_metrics(pred_4step, gold, scope_p, scope_r)
        row = {'model': model_label, 'method': '4-step_optimized', 'log': log_name, 'iteration': 'best'}
        row.update(m)
        all_rows.append(row)

        # Single prompt (5 iterations)
        single_preds = load_single_iterations(single_dir, log_name)
        for it_idx, pred in enumerate(single_preds):
            m = evaluate_all_metrics(pred, gold, scope_p, scope_r)
            row = {'model': model_label, 'method': 'single_prompt', 'log': log_name, 'iteration': it_idx + 1}
            row.update(m)
            all_rows.append(row)

    print(f'  Done: {model_label}')

df = pd.DataFrame(all_rows)
print(f'\nTotal rows: {len(df)}')
df.head()


GPT-5: 47 conversations
  Done: GPT-5

GPT-4o: 47 conversations
  Done: GPT-4o


Total rows: 846


,model,method,log,iteration,F1_Participants,F1_Restaurants,Final_Restaurant_Match,F1_Suggestion,F1_Response,F1_Mentioned,F1_PerceptionTable,F1_InterpretationTable
0,GPT-5,4-step_optimized,10_log,best,1.0,1.0,1.0,0.666667,0.666667,1.0,0.800000,0.533333
1,GPT-5,single_prompt,10_log,1,1.0,1.0,1.0,0.666667,0.666667,1.0,0.758621,0.450000
2,GPT-5,single_prompt,10_log,2,1.0,1.0,1.0,0.666667,0.666667,1.0,0.758621,0.366667
3,GPT-5,single_prompt,10_log,3,1.0,1.0,1.0,0.666667,0.666667,1.0,0.758621,0.633333
4,GPT-5,single_prompt,10_log,4,1.0,1.0,1.0,0.666667,0.666667,1.0,0.758621,0.500000


## Summary: Comparison Table

In [7]:
# Average iterations per conversation
conv_avg = df.groupby(['model', 'method', 'log'])[METRIC_NAMES].mean().reset_index()

# Display comparison
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n{"="*100}')
    print(f'  {model}')
    print(f'{"="*100}')
    print(f'{"Metric":<30} | {"4-Step Optimized":>20} | {"Single Prompt":>20} | {"Delta":>12}')
    print(f'{"-"*90}')

    four = conv_avg[(conv_avg['model'] == model) & (conv_avg['method'] == '4-step_optimized')]
    single = conv_avg[(conv_avg['model'] == model) & (conv_avg['method'] == 'single_prompt')]

    for m in METRIC_NAMES:
        fm, fs = four[m].mean(), four[m].std()
        sm, ss = single[m].mean(), single[m].std()
        delta = sm - fm
        print(f'{m:<30} | {fm:.4f} (+-{fs:.4f}) | {sm:.4f} (+-{ss:.4f}) | {delta:>+.4f}')


  GPT-5
Metric                         |     4-Step Optimized |        Single Prompt |        Delta
------------------------------------------------------------------------------------------
F1_Participants                | 1.0000 (+-0.0000) | 1.0000 (+-0.0000) | +0.0000
F1_Restaurants                 | 0.9981 (+-0.0133) | 0.9992 (+-0.0053) | +0.0012
Final_Restaurant_Match         | 0.9787 (+-0.1459) | 0.9617 (+-0.1360) | -0.0170
F1_Suggestion                  | 0.8323 (+-0.1907) | 0.6480 (+-0.2142) | -0.1843
F1_Response                    | 0.5369 (+-0.2907) | 0.4204 (+-0.2793) | -0.1165
F1_Mentioned                   | 0.9804 (+-0.0639) | 0.9777 (+-0.0597) | -0.0027
F1_PerceptionTable             | 0.8834 (+-0.0923) | 0.8422 (+-0.0993) | -0.0412
F1_InterpretationTable         | 0.4340 (+-0.2484) | 0.3764 (+-0.2059) | -0.0576

  GPT-4o
Metric                         |     4-Step Optimized |        Single Prompt |        Delta
----------------------------------------------------------

## Pivot Table

In [8]:
# Clean pivot table
pivot = conv_avg.groupby(['model', 'method'])[METRIC_NAMES].mean()
pivot = pivot.round(4)
display(pivot)

F1_Participants  F1_Restaurants  \
model     method                                              
GPT-4o 4-step_optimized              1.0          0.9884   
          single_prompt                 1.0          0.9908   
          single_prompt                 1.0          0.9739   
GPT-5     4-step_optimized              1.0          0.9981   
          single_prompt                 1.0          0.9992   

                            Final_Restaurant_Match  F1_Suggestion  \
model     method                                                    
GPT-4o 4-step_optimized                  0.9574         0.7411   
          single_prompt                     0.9191         0.6811   
          single_prompt                     0.9106         0.6044   
GPT-5     4-step_optimized                  0.9787         0.8323   
          single_prompt                     0.9617         0.6480   

                            F1_Response  F1_Mentioned  F1_PerceptionTable  \
model     method                                                            
GPT-4o 4-step_optimized       0.3365        0.9615              0.8359   
          single_prompt          0.2818        0.8683              0.7732   
          single_prompt          0.3339        0.7668              0.7030   
GPT-5     4-step_optimized       0.5369        0.9804              0.8834   
          single_prompt          0.4204        0.9777              0.8422   

                            F1_InterpretationTable  
model     method                                    
GPT-4o 4-step_optimized                  0.3675  
          single_prompt                     0.2807  
          single_prompt                     0.2564  
GPT-5     4-step_optimized                  0.4340  
          single_prompt                     0.3764

## Save Results

In [9]:
df.to_csv('appendix/results/comparison/unified_eval_detail.csv', index=False)
conv_avg.to_csv('appendix/results/comparison/unified_eval_by_conv.csv', index=False)

summary = conv_avg.groupby(['model', 'method'])[METRIC_NAMES].agg(['mean', 'std']).reset_index()
summary.to_csv('appendix/results/comparison/unified_eval_summary.csv', index=False)

print('Saved:')
print('  - appendix/results/comparison/unified_eval_detail.csv (per iteration)')
print('  - appendix/results/comparison/unified_eval_by_conv.csv (per conversation avg)')
print('  - appendix/results/comparison/unified_eval_summary.csv (model x method summary)')

Saved:
  - unified_eval_detail.csv (per iteration)
  - unified_eval_by_conv.csv (per conversation avg)
  - unified_eval_summary.csv (model x method summary)
